***We recommend working with this notebook on Google colab***
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ridatadiscoverycenter/riddc-jbook/blob/feature-volume-converter/riddc/notebooks/fox-kemper/Volume_viewer_converter.ipynb)

# Visualize osom data using the 3D volume viewer


<img src="https://github.com/ridatadiscoverycenter/riddc-jbook/raw/feature-volume-converter/riddc/notebooks/fox-kemper/images/volume-viewer-2.png" >

We have created the following example on how to export osom datasets and visualize them in 3D using the volume viewer application.
These scripts create large amount of data, and we recommend to use your google drive to save them. All the instructions here explains how to proceed.

You can use the volume viewer either from you computer or using Brown's Oscar HPC.

## 1 Get the Volume Viewer Application

### a. Download Volume Viewer

You can find the latest version of the volume viewer application on [this link](https://github.com/brown-ccv/VR-Volumeviewer/releases). Download the file `volume-viewer.zip` and unzip it on your local machine.


### b. Using Oscar HPC

Loggin in to Oscar using [ood](https://ood.ccv.brown.edu/) and open the desktop application with at least 1 GPU.

<img src="https://github.com/ridatadiscoverycenter/riddc-jbook/raw/feature-volume-converter/riddc/notebooks/fox-kemper/images/ood-desktop-session.png" >

Go to terminal and type the commands:

`module load osom-volume-viewer/1.0.1`

`VR-Volumeviewer`

## 2. Mount Google Drive

Execute the command below to connect to our google drive

**NOTE: The following command will open a pop-up window requesting access to your google drive file system**



In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

Optional: The previous code works most of the times. If you experience any issues and want to verify the book can see your google drive file system, execute the following code. It should display a list of files located in your google drive.

In [ ]:
%ls gdrive/MyDrive


## 3. Download Grid File

Now, we need to download the nc grid file to map our global data points to the application coordinates. Execue the following command.


In [ ]:
!wget https://github.com/brown-ccv/volume-data-converter/raw/main/volume_data_converter/resources/grid-data/osom_grid4_mindep_smlp_mod7-netcdf4.nc

##4. Install Volume Converter package

We have to install the `volume_data_converter` library that converts the osom nc files to volume viewer data

In [ ]:
! pip install volume-data-converter@git+https://github.com/brown-ccv/volume-data-converter.git

## 4. Download Osom data from ERDDAP

First we need to acquire data from the erddap server. In order to configure a connection and set up constrains for your search, create a json containing the connection url, protocol type and dataset id. You can also specify a time range and coordinates. The following is an example on how to create a json file with the mentioned properties. 
The command will create a file in your virtual local folder. We will use the result output in the next step.

In [1]:
import json 
dictionary ={ 
  "erddap_connection": {"server": "https://pricaimcit.services.brown.edu/erddap","protocol": "griddap","dataset_id": "model_data_57db_4a85_81d9"}, 
  "erddap_constrainsts": {"variables":["SalinityBottom","SalinitySurface"],  "time>=": "2019-12-30T00:00:00Z",
        "time<=": "2019-12-31T00:00:00Z",
        "time_step": 1,
        "eta_rho>=": 0,
        "eta_rho<=": 1099,
        "eta_rho_step": 1,
        "xi_rho>=": 0,
        "xi_rho<=": 999,
        "xi_rho_step": 1
        }, 
} 

with open('erddap_connection.json', 'w') as f:
    json_object = json.dumps(dictionary, indent = 4) 
    f.write(json_object)

After running the previous script, if we list the files in our local virtual folder, we can see the recently created `'erddap_connection.json'` file.


In [ ]:
%ls

Using the `volume_data_converter` library, we will pass `erddap_connection.json` to the `my_erddap_query` function to start up the to download process and retrieve the data from erddap.

In [ ]:
from volume_data_converter.erddap_query import erddap_query as my_erddap_query
my_erddap_query("./","erddap_connection.json")

If we list the files in our local virtual folder, we can see the ouput file `model_data_57db_4a85_81d9.nc` which is the name of the dataset id pointent in the `erddap_connection.json`.


In [ ]:
%ls

## 5. Convert NC-Osom data file to Volume viewer file format

Now we will create another json file containing the configuration to convert the nc file to raw data that the volume viewer app will read. The following code is an example of the type of configuration file the `volume_data_converter`expects to work with:

- `osom_grid_file`: Path to the nc grid file (the first downloaded file)
- `osom_data_file`: Path to the nc osom data file (`model_data_57db_4a85_81d9.nc`)
- `ouput_folder`: Path where the resulting package will be saved (a place in your google drive file system)
- `data_descriptor`: The osom variable to export to the voluem viewer (temp,salt)
- `time_frames`: If the dataset has multiple time frames, it will export all of them (time sequence). Otherwise, specify the required frame. (i.e: Null, [0,1],[5,6,7])
- `layer`: Currently osom supported files have 3 different types of layes `all`(default), `surface`, `bottom`.

The following code will create the `volume_viewer_data.json` file with the given configuration.


In [5]:
import json 
dictionary ={ 
  "osom_grid_file": "./osom_grid4_mindep_smlp_mod7-netcdf4.nc",
  "osom_data_file": "./model_data_57db_4a85_81d9.nc",
  "output_folder": "./gdrive/MyDrive/osom",
  "data_descriptor" : "SalinityBottom",
  "time_frames": [0],
  "layer" : "bottom"
} 

with open('volume_viewer_data.json', 'w') as f:
    json_object = json.dumps(dictionary, indent = 4) 
    f.write(json_object)

 Finally, we import the `create_osom_data` script and pass the `volume_viewer_data.json` file to start up the conversion process.

In [ ]:
from volume_data_converter.create_volume_viewer_data import create_osom_data as my_create_volume_viewer_data
my_create_volume_viewer_data("./volume_viewer_data.json")

Take in count the defined ouput file path points to `./gdrive/myDrive/osom`. That is a location inside **your** google drive. From there you can downloaded to the environment you are running the volume viewer application

In [9]:
%ls ./gdrive/MyDrive/osom/osom-data-SalinityBottom

data/  labels/  osom-loader.txt  textures/


## 6. Load data into the volume viewer


- Go to your google drive web app. In the left side panel you will find the result output folder (see image below). Do right click on the folder and select **Download**.

![google drive folder](https://github.com/ridatadiscoverycenter/riddc-jbook/raw/feature-volume-converter/riddc/notebooks/fox-kemper/images/Screen%20Shot%20google%20drive%20file%20system.png)


- Unzip the file and open the Volume viewer application.

- Click on `load file` and navegate to the root volume viewer data folder. Double click the `osom-loader.txt` file.

![volume viewer clip](https://github.com/ridatadiscoverycenter/riddc-jbook/raw/feature-volume-converter/riddc/notebooks/fox-kemper/images/clip-volume-viewer.gif)



- After a few seconds, the volume will be loaded.

![volume viewer clip](https://github.com/ridatadiscoverycenter/riddc-jbook/raw/feature-volume-converter/riddc/notebooks/fox-kemper/images/volume-viewer-4.png)


